In [0]:
%pip install openpyxl

In [0]:
%sql
select distinct sheet_name
from sandbox.z_yeswook_kim.jira_1410_raw

-- %sql
-- create or replace table sandbox.z_yeswook_kim.jira_1410_l1
-- select 
--     sheet_name,
--     row_number,
--     CSN,
--     MAC_ADDRESS,
--     concat_ws(':',
--         substring(MAC_ADDRESS, 1, 2),
--         substring(MAC_ADDRESS, 3, 2),
--         substring(MAC_ADDRESS, 5, 2),
--         substring(MAC_ADDRESS, 7, 2),
--         substring(MAC_ADDRESS, 9, 2),
--         substring(MAC_ADDRESS, 11, 2)
--     ) as mac_addr_origin,
--     kic_data_ods.tlamp.hash_mac(mac_addr_origin) as mac_addr
-- from sandbox.z_yeswook_kim.jira_1410_raw
-- -- group by 
--     -- sheet_name

In [0]:
%sql
select sheet_name, count(1)
from sandbox.z_yeswook_kim.jira_1410_raw
group by sheet_name

In [0]:
%sql
SELECT
    sheet_name,
    count(1)                        AS total_rows,
    count(DISTINCT row_number)      AS unique_rows,
    count(1) - count(DISTINCT row_number) AS duplicate_count
FROM sandbox.z_yeswook_kim.jira_1410_raw
WHERE sheet_name IN ('MA1', 'MA2', 'MA3', 'MA4')
GROUP BY sheet_name
ORDER BY sheet_name

In [0]:
%sql
# 1. 중복 제거된 MA2/3/4 임시 테이블 저장
spark.sql("""
    CREATE OR REPLACE TABLE sandbox.z_yeswook_kim.jira_1410_raw_temp AS
    SELECT DISTINCT AFFILIATE_CODE, FROM_ORGANIZATION_CODE, PROD_DATE, PROD_MODEL,
           WORK_ORDER, PROD_LINE, CSN, MAC_ADDRESS, sheet_name, row_number
    FROM sandbox.z_yeswook_kim.jira_1410_raw
    WHERE sheet_name IN ('MA2', 'MA3', 'MA4')
""")
print("1/4 임시 테이블 생성 완료")

# 2. 원본에서 MA2/3/4 전체 삭제
spark.sql("""
    DELETE FROM sandbox.z_yeswook_kim.jira_1410_raw
    WHERE sheet_name IN ('MA2', 'MA3', 'MA4')
""")
print("2/4 MA2/3/4 삭제 완료")

# 3. 중복 제거된 데이터 재적재
spark.sql("""
    INSERT INTO sandbox.z_yeswook_kim.jira_1410_raw
    SELECT * FROM sandbox.z_yeswook_kim.jira_1410_raw_temp
""")
print("3/4 재적재 완료")

# 4. 임시 테이블 정리
spark.sql("DROP TABLE IF EXISTS sandbox.z_yeswook_kim.jira_1410_raw_temp")
print("4/4 임시 테이블 삭제 완료 — 중복 제거 완료")

In [0]:
%sql
SELECT sheet_name, count(1) AS cnt
FROM sandbox.z_yeswook_kim.jira_1410_raw
GROUP BY sheet_name
ORDER BY sheet_name

In [0]:
# import pandas as pd
# import openpyxl

# wb = openpyxl.load_workbook("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", read_only=True)
# sheet_names = wb.sheetnames
# display(sheet_names)

# for sheet_name in sheet_names:
#     df_sheet = pd.read_excel("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", sheet_name=sheet_name)
#     df_sheet['sheet_name'] = sheet_name
#     df_sheet['row_number'] = range(1, len(df_sheet) + 1)
#     spark_df = spark.createDataFrame(df_sheet)
#     spark_df.write.format("delta").mode("append").saveAsTable("sandbox.z_yeswook_kim.jira_1410_raw")

In [0]:
sheet_name

In [0]:
import pandas as pd
import openpyxl

wb = openpyxl.load_workbook("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", read_only=True)
sheet_names = wb.sheetnames
display(sheet_names)

for sheet_name in ('MA2','MA3','MA4'):
    df_sheet = pd.read_excel("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", sheet_name=sheet_name)
    df_sheet['sheet_name'] = sheet_name
    df_sheet['row_number'] = range(1, len(df_sheet) + 1)
    spark_df = spark.createDataFrame(df_sheet)
    spark_df.write.format("delta").mode("append").saveAsTable("sandbox.z_yeswook_kim.jira_1410_raw")

In [0]:
%sql
create or replace table sandbox.z_yeswook_kim.jira_1410_l1
select 
    sheet_name,
    row_number,
    CSN,
    MAC_ADDRESS,
    concat_ws(':',
        substring(MAC_ADDRESS, 1, 2),
        substring(MAC_ADDRESS, 3, 2),
        substring(MAC_ADDRESS, 5, 2),
        substring(MAC_ADDRESS, 7, 2),
        substring(MAC_ADDRESS, 9, 2),
        substring(MAC_ADDRESS, 11, 2)
    ) as mac_addr_origin,
    eic_data_ods.tlamp.hash_mac(mac_addr_origin) as mac_addr
from sandbox.z_yeswook_kim.jira_1410_raw
-- group by 
    -- sheet_name

In [0]:
%sql
-- 프로토타입: 10건만 테스트 (normal_log는 webos60만 사용)
WITH base AS (
    SELECT *
    FROM sandbox.z_yeswook_kim.jira_1410_l1
    LIMIT 10
),

-- activation_date: 기기별 최초 crt_date, 마지막 last_chg_date
act AS (
    SELECT 
        mac_addr,
        MIN(crt_date) AS crt_date,
        MAX(last_chg_date) AS last_chg_date
    FROM eic_data_ods.tlamp.activation_date
    WHERE mac_addr IN (SELECT mac_addr FROM base)
    GROUP BY mac_addr
),

-- normal_log: 기기별 최대 log_create_time, 최대 accum_run_time
nl AS (
    SELECT 
        mac_addr,
        MAX(log_create_time) AS log_create_time,
        MAX(accum_run_time) AS accum_run_time
    FROM eic_data_ods.tlamp.normal_log_webos24
    WHERE context_name = 'tvpowerd'
      AND message_id = 'NL_POWER_STATE'
      AND mac_addr IN (SELECT mac_addr FROM base)
      AND date_ym = '2026-02'
    GROUP BY mac_addr
)

SELECT 
    b.*,
    act.mac_addr, nl.mac_addr,
    act.crt_date,
    act.last_chg_date,
    nl.log_create_time,
    nl.accum_run_time
FROM base b
LEFT JOIN act ON b.mac_addr = act.mac_addr
LEFT JOIN nl ON b.mac_addr = nl.mac_addr
where crt_date is not null


In [0]:
%sql
-- 전체 쿼리: 모든 normal_log 버전 포함
create or replace table sandbox.z_yeswook_kim.jira_1410_l2
WITH base AS (
    SELECT *
    FROM sandbox.z_yeswook_kim.jira_1410_l1
),

-- activation_date: 기기별 최초 crt_date, 마지막 last_chg_date
act AS (
    SELECT 
        mac_addr,
        MIN(crt_date) AS crt_date,
        MAX(last_chg_date) AS last_chg_date
    FROM eic_data_ods.tlamp.activation_date
    WHERE mac_addr IN (SELECT mac_addr FROM base)
    GROUP BY mac_addr
),

-- normal_log: 6개 버전 UNION ALL (tvpowerd + NL_POWER_STATE 한정)
nl AS (
    SELECT 
        mac_addr,
        MAX(log_create_time) AS log_create_time,
        MAX(accum_run_time) AS accum_run_time
    FROM (
        SELECT mac_addr, log_create_time, accum_run_time
        FROM eic_data_ods.tlamp.normal_log_webos60
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM eic_data_ods.tlamp.normal_log_webos22
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM eic_data_ods.tlamp.normal_log_webos23
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM eic_data_ods.tlamp.normal_log_webos24
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM eic_data_ods.tlamp.normal_log_webos25
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM eic_data_ods.tlamp.normal_log_webos26
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
    )
    GROUP BY mac_addr
)

SELECT 
    b.*,
    act.crt_date,
    act.last_chg_date,
    nl.log_create_time,
    nl.accum_run_time
FROM base b
LEFT JOIN act ON b.mac_addr = act.mac_addr
LEFT JOIN nl ON b.mac_addr = nl.mac_addr


In [0]:
%sql
    SELECT 
        *
        -- mac_addr,
        -- MIN(crt_date) AS crt_date,
        -- MAX(last_chg_date) AS last_chg_date
    FROM eic_data_ods.tlamp.activation_date
    -- WHERE mac_addr IN (SELECT mac_addr_origin FROM base)
    -- GROUP BY mac_addr

In [0]:
%sql
select 
    sheet_name, count(1), count(crt_date), count(last_chg_date), count(log_create_time), count(last_chg_date)
from sandbox.z_yeswook_kim.jira_1410_l2
group by sheet_name

In [0]:
from pyspark.sql import SparkSession

df = spark.table("sandbox.z_yeswook_kim.jira_1410_l2")
sheet_names = [row.sheet_name for row in df.select("sheet_name").distinct().collect()]

for sheet_name in sheet_names:
    df_sheet = df.filter(df.sheet_name == sheet_name)
    output_path = f"/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_l2_csv/{sheet_name}.csv"
    df_sheet.coalesce(1).write.option("header", "true").mode("overwrite").csv(output_path)